# Person B — BM25 sweep on Kaggle

Runs the full 5-strategy BM25 sweep on Kaggle CPU (T4×2 GPUs are unused; BM25 is Lucene-CPU only).

**Inputs:** preprocessed MIRACL-id from the public HF dataset `Maskrio/nya-ir-miracl-id-preprocessed` (2.82 GB).

**Outputs (under `/kaggle/working/results/`):**
- `runs/bm25/bm25__<strategy>.txt` — TREC run files
- `metrics/bm25/bm25__<strategy>.csv` — per-query metric CSVs
- `reports/bm25_forest.png` — forest plot of paired effect sizes vs Keep

**Runtime:** ~2–4 h end-to-end on Kaggle CPU; well within the 12 h interactive limit.

**Sanity gate:** halts after the Keep baseline if mean nDCG@10 misses the published MIRACL number (0.449) by more than ±0.02. Override with `--reference-ndcg10` / `--sanity-tolerance` if defensible.

## 1. Install JDK 21 + clone the repo + Pyserini

Pyserini wraps Lucene, so a JDK is mandatory. Kaggle's base image does not ship one.

In [ ]:
import os, sys

!apt-get update -qq && apt-get install -y openjdk-21-jdk-headless -qq
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-21-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:" + os.environ["PATH"]
!java -version

# Person B's orchestrator, sanity gate, and forest plot live on the BM25-Track branch.
!git clone --branch BM25-Track https://github.com/shaneemichael/TA_TBI.git /kaggle/working/repo
%cd /kaggle/working/repo
!pip install -e ".[bm25]" -q
# Force a new huggingface_hub so the `hf` CLI is present; Kaggle's preinstalled
# version (1.10.x) ships a broken `huggingface-cli` that silently no-ops.
!pip install --upgrade --quiet "huggingface_hub>=0.24"

## 2. Pull preprocessed MIRACL-id from HuggingFace

Public dataset, no login required. Downloads to `/kaggle/working/hf_data/` so the orchestrator can read directly.

In [ ]:
!hf download Maskrio/nya-ir-miracl-id-preprocessed \
  --repo-type dataset \
  --local-dir /kaggle/working/hf_data

!ls /kaggle/working/hf_data
!ls /kaggle/working/hf_data/keep | head
!wc -l /kaggle/working/hf_data/keep/queries_dev.jsonl /kaggle/working/hf_data/keep/corpus_train.jsonl /kaggle/working/hf_data/qrels/qrels_dev.txt

## 3. Run the BM25 sweep

Indices go to `/kaggle/temp/` (not persisted; freed at session end). Per-query CSVs go to `/kaggle/working/` so they are preserved when you commit the notebook.

The sweep runs Keep first, sanity-gates, then proceeds with `naive_strip`, `sastrawi_clitic`, `sentinel`, `rule_resolved`. Idempotent — safe to re-execute this cell after an interrupt.

In [ ]:
!python scripts/run_bm25_pipeline.py \
  --processed-dir /kaggle/working/hf_data \
  --qrels /kaggle/working/hf_data/qrels/qrels_dev.txt \
  --index-root /kaggle/temp/indexes/bm25 \
  --staging-dir /kaggle/temp/staging/bm25 \
  --run-dir /kaggle/working/results/runs/bm25 \
  --metric-dir /kaggle/working/results/metrics/bm25 \
  --threads 4

## 4. Forest plot — paired effect sizes vs Keep

Bootstrap 95 % CIs on mean ΔnDCG@10, Cliff's δ labelled on each marker.

In [ ]:
!python scripts/plot_forest.py \
  --baseline /kaggle/working/results/metrics/bm25/bm25__keep.csv \
  --treatment naive_strip=/kaggle/working/results/metrics/bm25/bm25__naive_strip.csv \
  --treatment sastrawi_clitic=/kaggle/working/results/metrics/bm25/bm25__sastrawi_clitic.csv \
  --treatment sentinel=/kaggle/working/results/metrics/bm25/bm25__sentinel.csv \
  --treatment rule_resolved=/kaggle/working/results/metrics/bm25/bm25__rule_resolved.csv \
  --output /kaggle/working/results/reports/bm25_forest.png

from IPython.display import Image
Image("/kaggle/working/results/reports/bm25_forest.png")

## 5. (Optional) inspect what landed

Quick sanity-check on the 5 metric CSVs before committing the notebook output.

In [ ]:
import pandas as pd
from pathlib import Path

metric_dir = Path("/kaggle/working/results/metrics/bm25")
summary = []
for csv_path in sorted(metric_dir.glob("bm25__*.csv")):
    df = pd.read_csv(csv_path)
    summary.append({
        "condition": csv_path.stem,
        "n_queries": len(df),
        "mean_ndcg@10": df["ndcg@10"].mean(),
        "mean_mrr@100": df["mrr@100"].mean(),
        "mean_recall@100": df["recall@100"].mean(),
    })
pd.DataFrame(summary)